In [ ]:
# Fusion des deux sources de données
# Source 1 : Kaggle Crop Recommendation Dataset (données réelles)
# Source 2 : Dataset simulé calibré FAO/CNRA CI (cultures ivoiriennes)

import pandas as pd
import numpy as np
import os

np.random.seed(42)

# Création des dossiers nécessaires s'ils n'existent pas
os.makedirs('../data', exist_ok=True)

# PARTIE 1 - CHARGEMENT ET FILTRAGE DU DATASET KAGGLE

df_kaggle = pd.read_csv('../data/Crop_recommendation.csv')

print(" Dataset Kaggle chargé ")
print(f"Shape initial : {df_kaggle.shape}")
print(f"Cultures disponibles : {df_kaggle['label'].unique().tolist()}")

cultures_ci_kaggle = ['coffee', 'maize', 'rice', 'cotton', 'coconut']
df_kaggle_ci = df_kaggle[df_kaggle['label'].isin(cultures_ci_kaggle)].copy()

print(f"\nAprès filtrage (cultures CI uniquement) : {df_kaggle_ci.shape}")
print(df_kaggle_ci['label'].value_counts())

# PARTIE 2 - GÉNÉRATION DES CULTURES MANQUANTES (FAO/CNRA CI)

N_SAMPLES = 100

def generate_crop(crop_name, n_min, n_max, p_mean, p_std,
                  k_mean, k_std, temp_mean, temp_std,
                  hum_mean, hum_std, ph_mean, ph_std,
                  rain_mean, rain_std):
    return pd.DataFrame({
        'N': np.random.randint(n_min, n_max, size=N_SAMPLES),
        'P': np.clip(np.random.normal(p_mean, p_std, N_SAMPLES).astype(int), 5, 145),
        'K': np.clip(np.random.normal(k_mean, k_std, N_SAMPLES).astype(int), 5, 205),
        'temperature': np.clip(np.round(np.random.normal(temp_mean, temp_std, N_SAMPLES), 2), 18.0, 42.0),
        'humidity': np.clip(np.round(np.random.normal(hum_mean, hum_std, N_SAMPLES), 2), 10.0, 100.0),
        'ph': np.clip(np.round(np.random.normal(ph_mean, ph_std, N_SAMPLES), 2), 3.5, 9.0),
        'rainfall': np.clip(np.round(np.random.normal(rain_mean, rain_std, N_SAMPLES), 2), 300.0, 3000.0),
        'label': [crop_name] * N_SAMPLES
    })

cultures_simulees = [
    # CACAO - Zone forestière Sud (Soubré, San-Pédro, Aboisso)
    # Source : CNRA CI programme cacao 2020-2023, CIRAD
    dict(crop_name='cacao',
         n_min=30, n_max=60, p_mean=55, p_std=10, k_mean=40, k_std=8,
         temp_mean=26.5, temp_std=1.5, hum_mean=82.0, hum_std=5.0,
         ph_mean=6.2, ph_std=0.4, rain_mean=1600.0, rain_std=180.0),

    # ANACARDE - Zone Nord sèche (Korhogo, Bondoukou, Mankono)
    # Source : ACA (African Cashew Alliance) 2023, FIRCA
    dict(crop_name='anacarde',
         n_min=15, n_max=35, p_mean=25, p_std=5, k_mean=30, k_std=6,
         temp_mean=30.0, temp_std=2.5, hum_mean=45.0, hum_std=8.0,
         ph_mean=5.8, ph_std=0.6, rain_mean=1000.0, rain_std=150.0),

    # IGNAME - Zone Centre et Ouest (Bouaké, Katiola, Man)
    # Source : FAO West Africa crop requirements, MINAGRI CI
    dict(crop_name='igname',
         n_min=50, n_max=90, p_mean=50, p_std=10, k_mean=80, k_std=15,
         temp_mean=27.5, temp_std=1.5, hum_mean=78.0, hum_std=6.0,
         ph_mean=5.5, ph_std=0.5, rain_mean=1300.0, rain_std=180.0),

    # PLANTAIN - Toutes zones humides (Agboville, Divo, Gagnoa)
    # Source : FAO Ecocrop, OCPV CI
    dict(crop_name='plantain',
         n_min=70, n_max=110, p_mean=65, p_std=12, k_mean=100, k_std=18,
         temp_mean=27.0, temp_std=1.8, hum_mean=80.0, hum_std=5.0,
         ph_mean=6.3, ph_std=0.4, rain_mean=1500.0, rain_std=190.0),
]

df_simule = pd.concat([generate_crop(**c) for c in cultures_simulees], ignore_index=True)

print("\n Dataset simulé généré (FAO/CNRA CI) ")
print(f"Shape : {df_simule.shape}")
print(df_simule['label'].value_counts())

# PARTIE 3 - VÉRIFICATION DE LA COHÉRENCE DES COLONNES

colonnes_attendues = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']

assert list(df_kaggle_ci.columns) == colonnes_attendues, \
    f"Colonnes Kaggle incorrectes : {list(df_kaggle_ci.columns)}"

assert list(df_simule.columns) == colonnes_attendues, \
    f"Colonnes simulées incorrectes : {list(df_simule.columns)}"

print("\n Colonnes cohérentes entre les deux sources")

# PARTIE 4 - FUSION ET EXPORT FINAL

df_final = pd.concat([df_kaggle_ci, df_simule], ignore_index=True)
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print("\n Dataset final")
print(f"Shape : {df_final.shape}")
print(f"\nDistribution par culture :")
print(df_final['label'].value_counts())
print(f"\nVérification valeurs manquantes :")
print(df_final.isnull().sum())
print(f"\nStatistiques générales :")
print(df_final.describe().round(2))

# Export
df_final.to_csv('../data/dataset_agricole_ci_final.csv', index=False)
print("\n Dataset exporté : data/dataset_agricole_ci_final.csv")
print(f"{df_final['label'].nunique()} cultures | {df_final.shape[0]} lignes total")

 Dataset Kaggle chargé 
Shape initial : (2200, 8)
Cultures disponibles : ['rice', 'maize', 'chickpea', 'kidneybeans', 'pigeonpeas', 'mothbeans', 'mungbean', 'blackgram', 'lentil', 'pomegranate', 'banana', 'mango', 'grapes', 'watermelon', 'muskmelon', 'apple', 'orange', 'papaya', 'coconut', 'cotton', 'jute', 'coffee']

Après filtrage (cultures CI uniquement) : (500, 8)
label
rice       100
maize      100
coconut    100
cotton     100
coffee     100
Name: count, dtype: int64

 Dataset simulé généré (FAO/CNRA CI) 
Shape : (400, 8)
label
cacao       100
anacarde    100
igname      100
plantain    100
Name: count, dtype: int64

 Colonnes cohérentes entre les deux sources

 Dataset final
Shape : (900, 8)

Distribution par culture :
label
rice        100
plantain    100
coconut     100
cacao       100
igname      100
maize       100
coffee      100
anacarde    100
cotton      100
Name: count, dtype: int64

Vérification valeurs manquantes :
N              0
P              0
K              0
te